In [1]:
# Load libraries
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("../../../data/data_all.csv")

In [3]:
# Subset data to station of interest
df = df[df["station"] == "Nongzhanguan"].copy()

# reset index
df = df.reset_index(drop=True)

In [4]:
df.head()

,Unnamed: 0,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,245449,2013,3,1,0,5.0,14.0,4.0,12.0,200,85.0,-0.5,1024.5,-21.4,0.0,NNW,5.7,Nongzhanguan
1,245450,2013,3,1,1,8.0,12.0,6.0,14.0,200,84.0,-0.7,1025.1,-22.1,0.0,NW,3.9,Nongzhanguan
2,245451,2013,3,1,2,3.0,6.0,5.0,14.0,200,83.0,-1.2,1025.3,-24.6,0.0,NNW,5.3,Nongzhanguan
3,245452,2013,3,1,3,5.0,5.0,5.0,14.0,200,84.0,-1.4,1026.2,-25.5,0.0,N,4.9,Nongzhanguan
4,245453,2013,3,1,4,5.0,5.0,6.0,21.0,200,77.0,-1.9,1027.1,-24.5,0.0,NNW,3.2,Nongzhanguan


R code was essentially TEMP + PRES + RAIN + WSPM * wd_coarse. `model.matrix()` automatically expands those categorical variables and interactions but in Python I have to do it myself.

## Build the Predictor Matrix X 

In [5]:
mapping = {
    "ENE": "E",
    "ESE": "E",
    "NNE": "NE",
    "NNW": "NW",
    "SSE": "SE",
    "SSW": "SW",
    "WNW": "W",
    "WSW": "W"
}

df["wd_coarse"] = df["wd"].replace(mapping)

In [6]:
df["wd_coarse"] = df["wd_coarse"].astype("category")

In [7]:
df["wd_coarse"].value_counts()

wd_coarse
E     9607
NE    5170
SW    4186
NW    3932
W     3855
SE    3120
N     1763
S     1481
Name: count, dtype: int64

In [8]:
# numeric predictors
X_num = df[["TEMP", "PRES", "RAIN", "WSPM"]]

# categorical variable
wd_dummies = pd.get_dummies(df["wd_coarse"], drop_first=False)

### Create Interaction term WSPM * winddirection

In [9]:
WSPM = df["WSPM"].values.reshape(-1,1)
interaction = WSPM * wd_dummies.values
interaction = pd.DataFrame(interaction,
                           columns=[f"WSPM_{c}" for c in wd_dummies.columns])

In [10]:
interaction.head()

,WSPM_E,WSPM_N,WSPM_NE,WSPM_NW,WSPM_S,WSPM_SE,WSPM_SW,WSPM_W
0,0.0,0.0,0.0,5.7,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,3.9,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,5.3,0.0,0.0,0.0,0.0
3,0.0,4.9,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,3.2,0.0,0.0,0.0,0.0


### Combine Everything

In [11]:
X_df = pd.concat([X_num, wd_dummies, interaction], axis=1)

X = X_df.values

In [12]:
X = X_df.to_numpy(dtype=np.float32)

## Create Y

In [13]:
Y = np.log(df[["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]].values)

In [14]:
Y

array([[1.60943791, 2.63905733, 1.38629436, 2.48490665, 5.29831737,
        4.44265126],
       [2.07944154, 2.48490665, 1.79175947, 2.63905733, 5.29831737,
        4.4308168 ],
       [1.09861229, 1.79175947, 1.60943791, 2.63905733, 5.29831737,
        4.41884061],
       ...,
       [2.7080502 , 3.66356165, 1.60943791, 3.91202301, 6.39692966,
        3.71357207],
       [2.39789527, 3.55534806, 1.79175947, 3.8918203 , 6.2146081 ,
        3.71357207],
       [2.30258509, 3.33220451, 1.94591015, 3.87120101, 6.39692966,
        3.66356165]])

## Index Split

In [15]:
print(X.shape[0])
print(len(df))

33114
33114


In [16]:
# Index Split
n = X.shape[0]

train_idx = np.arange(0, n, 3)
valid_idx = np.arange(1, n, 3)
test_idx  = np.arange(2, n, 3)

In [17]:
# Apply the index split
X_train = X[train_idx]
X_valid = X[valid_idx]
X_test  = X[test_idx]

Y_train = Y[train_idx]
Y_valid = Y[valid_idx]
Y_test  = Y[test_idx]

## Convert to Tensors for PyTorch

In [18]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_valid_tensor = torch.tensor(X_valid, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
Y_valid_tensor = torch.tensor(Y_valid, dtype=torch.float32)
Y_test_tensor  = torch.tensor(Y_test, dtype=torch.float32)

In [19]:
print(X_train_tensor.shape)
print(Y_train_tensor.shape)

torch.Size([11038, 20])
torch.Size([11038, 6])


## Neural Network



In [20]:
learning_rates = [1e-4, 5e-4, 1e-3]

In [21]:
results = []

### Build model with different learning rates

In [22]:
for lr in learning_rates:

    model = nn.Sequential(
        nn.Linear(X_train.shape[1], 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 6)
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(200):

        preds = model(X_train_tensor)
        loss = loss_fn(preds, Y_train_tensor)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # validation evaluation
    with torch.no_grad():
        val_preds = model(X_valid_tensor)
        val_mse = loss_fn(val_preds, Y_valid_tensor).item()

    results.append((lr, val_mse))

In [23]:
results

[(0.0001, 1.1708184480667114),
 (0.0005, 1.1221405267715454),
 (0.001, 1.1091467142105103)]

In [24]:
# Get the best learning rate
best_lr, best_val_mse = min(results, key=lambda x: x[1])

print("Best learning rate:", best_lr)
print("Validation MSE:", best_val_mse)

Best learning rate: 0.001
Validation MSE: 1.1091467142105103


### Retrain the model with the best learning rate

In [25]:

model = nn.Sequential(
     nn.Linear(X_train.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 6)
)

optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)

for epoch in range(200):

    preds = model(X_train_tensor)
    loss = loss_fn(preds, Y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

### Evaluate model on test set

In [26]:
with torch.no_grad():
    test_preds = model(X_test_tensor)
    test_mse = loss_fn(test_preds, Y_test_tensor).item()

# # Same as above but without a context manager
# test_preds = model(X_test_tensor).detach().numpy()
# mse = mean_squared_error(Y_test, test_preds)

print("Final test MSE:", test_mse)

Final test MSE: 1.0700223445892334
